In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
%run ../../../utils/reader_utils

In [0]:
%run ../../../utils/writer_utils

In [0]:
# Constants

SOURCE_TABLE_CONF = {
    "table": "orders_raw",
    "schema": "bronze",
}

TARGET_TABLE_CONF = {
    "table": "pyspark_scd2_orders",
    "schema": "silver",
}

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today()

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today + timedelta(days=1)
CATALOG = f"sl_{ENV}"
CHECKPOINT_BASE = f"""/Volumes/{CATALOG}/{TARGET_TABLE_CONF.get("schema")}/checkpoints"""

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
source_table = f"""{SOURCE_TABLE_CONF.get("schema")}.{SOURCE_TABLE_CONF.get("table")}"""
target_table = f"""{TARGET_TABLE_CONF.get("schema")}.{TARGET_TABLE_CONF.get("table")}"""
checkpoint_path = f"""{CHECKPOINT_BASE}/{TARGET_TABLE_CONF.get("table")}/"""
print(source_table, target_table, checkpoint_path)

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    import shutil
    try:
        shutil.rmtree(checkpoint_path)
    except FileNotFoundError:
        print("no chekpoints found")

    empty_df = (spark.createDataFrame([], spark.read.table(source_table).schema)
      .withColumn("_change_ts", F.lit(None).cast(TimestampType()))
      .withColumn("customer_id", F.lit(None).cast(IntegerType()))
      .withColumn("delivery_date", F.lit(None).cast(DateType()))
      .withColumn("order_date", F.lit(None).cast(DateType()))
      .withColumn("quantity", F.lit(None).cast(IntegerType()))
      .withColumn("total_amount", F.lit(None).cast(DoubleType()))
      .withColumn("start_at", F.lit(None).cast(TimestampType()))
      .withColumn("end_at", F.lit(None).cast(TimestampType()))
      .withColumn("is_current", F.lit(None).cast(BooleanType()))
      .withColumn("_insert_ts", F.lit(None).cast(TimestampType()))
      .withColumn("_update_ts", F.lit(None).cast(TimestampType()))
      .select("order_id",
                "customer_id",
                "delivery_date",
                "destination_city",
                "order_date",
                "origin_city",
                "payment_method",
                "product_code",
                "product_name",
                "quantity",
                "status",
                "total_amount",
                "start_at",
                "end_at",
                "is_current",
                "_insert_ts",
                "_update_ts")
      )

    (empty_df
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY (order_id, start_at, end_at)")

In [0]:
bronze_orders_df = (
    spark.readStream
    .option("startingVersion", 0)
    .table(source_table)
    .filter(F.col("order_id").isNotNull())
    .withColumn("_process_ts", F.current_timestamp())
    .withColumn("_change_ts", F.col("_change_ts").cast(TimestampType()))
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("delivery_date", F.col("delivery_date").cast(DateType()))
    .withColumn("order_date", F.col("order_date").cast(DateType()))
    .withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("total_amount", F.col("total_amount").cast(DoubleType()))
    )

In [0]:
def merge_to_scd2_orders(batch_df, batch_id):
    target = DeltaTable.forName(spark, target_table)

    # refine batch_df, and account for multiple records in same batch per customer_id
    window_spec = Window.partitionBy("order_id").orderBy("_change_ts")
    refined_batch_df = (batch_df
                        .withColumn("start_at", F.col("_change_ts"))
                        .withColumn("next_start", F.lead(F.col("_change_ts")).over(window_spec))
                        .withColumn("end_at", F.expr("timestampadd(MILLISECOND, -1, next_start)"))
                        .withColumn("is_current", F.col("next_start").isNull())
                        .withColumn("sequence_order", F.row_number().over(window_spec))
    )

    # close current rows that have changed matched on key column and change_ts for idempotency
    (target.alias("t").merge(
      refined_batch_df.filter(F.col("sequence_order") == 1).alias("s"),
      "s.order_id = t.order_id and t.is_current and s._change_ts > t._change_ts")
    .whenMatchedUpdate(set = {"end_at": F.expr("timestampadd(MILLISECOND, -1, s.start_at)"),
                              "is_current": F.lit(False),
                              "_update_ts": "s._process_ts"})
    .execute()
    ) 
    
    # insert new rows matched on key column and change_ts for idempotency
    (target.alias("t").merge(
      refined_batch_df.alias("s"),
      "s.order_id = t.order_id and s._change_ts = t._change_ts")
    .whenNotMatchedInsert(values = {"customer_id": "s.customer_id",
                              "delivery_date": "s.delivery_date",
                              "destination_city": "s.destination_city",
                              "order_date": "s.order_date",
                              "order_id": "s.order_id",
                              "origin_city": "s.origin_city",
                              "payment_method": "s.payment_method",
                              "product_code": "s.product_code",
                              "product_name": "s.product_name",
                              "quantity": "s.quantity",
                              "total_amount": "s.total_amount",
                              "start_at": "s.start_at",
                              "end_at": "s.end_at",
                              "is_current": "s.is_current",
                              "_insert_ts": "s._process_ts"})
    .execute()
    ) 

In [0]:
query = (bronze_orders_df.writeStream
  .trigger(availableNow=True)
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .foreachBatch(merge_to_scd2_orders)
  .start()
)

query.awaitTermination()